# PyTorch 模型

In [1]:
import sys
sys.path.extend([".."])
import set_env

ROOT: /media/pc/data/lxw/ai/tvm-book


In [2]:
from tvm_book.tools.frontends import Frontend, TrainInputConfig
from tvm_book.tools import display

caffe 前端模型配置：

In [3]:
print(display.Tree("| ")("models/pytorch_demo"))

| caffe_demo
    | config.toml
    | test.caffemodel
    | test.prototxt


{icon}`fa fa-folder-open` `caffe_demo/` 文件夹下存在如下内容：

- {icon}`fa fa-file` `test.caffemodel` 存储 caffe 模型参数的初始化模型
- {icon}`fa fa-file` `test.prototxt` 存储 caffe 模型结构
- {icon}`fa fa-file` `config.toml` 存储 caffe 模型配置信息
    ```{include} models/caffe_demo/config.toml
    :code: toml
    ```

In [26]:
import toml

model_root = "models/caffe_demo"

config = toml.load(f"{model_root}/config.toml")

frontend = Frontend(config["model"]["model_type"])
if len(config["train_inputs"]) == 1: # "此模型为单输入模型"
    input_config = TrainInputConfig(**config["train_inputs"][0])
    shape_dict = {input_config.name: input_config.shape}
    dtype_dict = {input_config.name: input_config.dtype}

model = frontend.load(
    f"{model_root}/{config['model']['init_net_path']}", 
    predict_net_path=f"{model_root}/{config['model']['predict_net_path']}", 
    shape_dict=shape_dict, 
    dtype_dict=dtype_dict
)

In [27]:
import tvm
from tvm import relay

with tvm.transform.PassContext(opt_level=3, disabled_pass={"AlterOpLayout"}):
    mod = relay.quantize.prerequisite_optimize(model.mod, model.params)

The Relay type checker is unable to show the following types match:
  Tensor[(1), float32]
  Tensor[(64), float32]
In particular:
  dimension 0 conflicts: 1 does not match 64.
The Relay type checker is unable to show the following types match.
In particular `Tensor[(64), float32]` does not match `Tensor[(1), float32]`
The Relay type checker is unable to show the following types match:
  Tensor[(1), float32]
  Tensor[(64), float32]
In particular:
  dimension 0 conflicts: 1 does not match 64.
The Relay type checker is unable to show the following types match.
In particular `Tensor[(64), float32]` does not match `Tensor[(1), float32]`
The Relay type checker is unable to show the following types match:
  Tensor[(1), float32]
  Tensor[(64), float32]
In particular:
  dimension 0 conflicts: 1 does not match 64.
The Relay type checker is unable to show the following types match.
In particular `Tensor[(64), float32]` does not match `Tensor[(1), float32]`
The Relay type checker is unable to show

DiagnosticError: Traceback (most recent call last):
  8: tvm::runtime::PackedFuncObj::Extractor<tvm::runtime::PackedFuncSubObj<tvm::runtime::TypedPackedFunc<tvm::IRModule (tvm::transform::Pass, tvm::IRModule)>::AssignTypedLambda<tvm::transform::__mk_TVM9::{lambda(tvm::transform::Pass, tvm::IRModule)#1}>(tvm::transform::__mk_TVM9::{lambda(tvm::transform::Pass, tvm::IRModule)#1}, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >)::{lambda(tvm::runtime::TVMArgs const&, tvm::runtime::TVMRetValue*)#1}> >::Call(tvm::runtime::PackedFuncObj const*, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >, tvm::runtime::TVMRetValue)
  7: tvm::transform::Pass::operator()(tvm::IRModule) const
  6: tvm::transform::Pass::operator()(tvm::IRModule, tvm::transform::PassContext const&) const
  5: tvm::transform::SequentialNode::operator()(tvm::IRModule, tvm::transform::PassContext const&) const
  4: tvm::transform::Pass::operator()(tvm::IRModule, tvm

In [20]:
print(model.mod)

def @main(%input.1: Tensor[(1, 3, 192, 160), float32]) {
  %0 = nn.conv2d(%input.1, meta[relay.Constant][0], strides=[2, 2], padding=[1, 1, 1, 1], channels=16, kernel_size=[3, 3]);
  %1 = nn.bias_add(%0, meta[relay.Constant][1]);
  %2 = nn.prelu(%1, meta[relay.Constant][2]);
  %3 = nn.conv2d(%2, meta[relay.Constant][3], strides=[2, 2], padding=[1, 1, 1, 1], channels=32, kernel_size=[3, 3]);
  %4 = nn.bias_add(%3, meta[relay.Constant][4]);
  %5 = nn.prelu(%4, meta[relay.Constant][5]);
  %6 = nn.conv2d(%5, meta[relay.Constant][6], padding=[0, 0, 0, 0], channels=32, kernel_size=[1, 1]);
  %7 = nn.bias_add(%6, meta[relay.Constant][7]);
  %8 = nn.prelu(%7, meta[relay.Constant][8]);
  %9 = nn.conv2d(%8, meta[relay.Constant][9], padding=[0, 0, 0, 0], channels=8, kernel_size=[1, 1]);
  %10 = nn.bias_add(%9, meta[relay.Constant][10]);
  %11 = nn.relu(%10);
  %12 = nn.conv2d(%11, meta[relay.Constant][11], padding=[1, 1, 1, 1], groups=8, channels=8, kernel_size=[3, 3]);
  %13 = nn.bias_add(%12, m